# [모델] LightGBM + CatBoost 앙상블 — 추론

학습 노트북이 저장한 모델(`../model/ensemble.pkl`)을 불러와 평가 데이터
(`../data/test.csv`)를 예측하고, 제출 파일(`../output/submission.csv`)을 만듭니다.
(이 노트북은 `notebooks/` 아래에 있어 저장소 루트 기준 경로가 `../` 로 시작합니다.)

**코드 제출 방식** — 이 대회는 결과 CSV 가 아니라 코드를 제출합니다. 아래 구조의 zip 을
제출하면 평가 서버가 zip 을 풀고 `script.py` 를 그대로 실행하여 채점합니다.

```
submit.zip
├── model/
│   └── ensemble.pkl         # LightGBM+CatBoost 모델, 로지스틱 회귀 메타러너(스태킹), 피처목록 딕셔너리
├── script.py               # 추론 코드 (서버가 실행) — src/script.py 와 동일 코드
└── requirements.txt        # 필요한 라이브러리 (lightgbm, catboost 포함) — src/requirements.txt
```

zip 안에서는 `model/`, `script.py` 가 같은 위치에 있으므로 `src/script.py` 는 `./model`
처럼 경로가 한 단계 짧습니다 — 이 노트북의 셀을 `src/script.py` 로 옮길 때는 경로 접두사를
`../` 에서 `./` 로 바꿔야 합니다 (이미 반영되어 있습니다).

배포된 `test.csv` 는 제출 형식 확인용 5행 샘플입니다. 실제 평가 데이터 245,789행은
평가 서버에만 있으며, 서버가 같은 경로(`./data/test.csv`)로 넣어 줍니다.

## 1. 라이브러리 불러오기

`pandas`/`numpy` 로 데이터를 읽고 피처를 만들고 결과를 씁니다. `joblib` 으로 학습 때
저장한 모델 아티팩트(딕셔너리)를 불러옵니다.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd

ID_COL = "row_id"
TARGET_COL = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]

## 2. 데이터 로드와 피처 엔지니어링

추론 입력은 반드시 **학습 때와 똑같은 방식**으로 만들어야 합니다. LightGBM은 범주형과
결측값을 자체적으로 처리하지만, 파생 피처(`build_features`)는 파이프라인 밖에서 만든
일반 파이썬 코드이므로 학습 노트북의 코드와 한 글자도 다르지 않게 여기 다시 넣습니다.

모든 파생 피처는 현재 행의 값만 사용하므로, 평가 데이터의 각 행을 독립적으로 예측한다는
대회 규칙을 지킵니다.

In [ ]:
# =======================
# 데이터 로드 유틸
# =======================

def load_test(path):
    """평가 데이터(csv) 로드. 한 행이 투구 하나."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    if ID_COL not in df.columns:
        raise ValueError(f"test 데이터에 {ID_COL} 컬럼이 없음: {list(df.columns)[:5]}")
    return df


def load_sample_submission(path):
    """sample_submission.csv 로드 — 제출 파일의 row_id 순서/컬럼 기준."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    if list(df.columns[:2]) != [ID_COL, TARGET_COL]:
        raise ValueError(
            f"sample_submission 컬럼이 ({ID_COL}, {TARGET_COL})이 아님: "
            f"{list(df.columns)}")
    return df


# =======================
# 학습 때 사용한 피처 엔지니어링 (그대로) — 학습 노트북의 build_features()와 동일해야 함
# =======================

def build_features(df, global_mean, tk_lookup):
    """모델 입력 피처 생성. row_id를 제외한 원본 컬럼 + 파생 피처를 만든다."""
    df = df.copy()

    df["is_two_strike"] = (df["strikes_before"] >= 2).astype(int)
    df["is_three_ball"] = (df["balls_before"] >= 3).astype(int)
    df["is_full_count"] = ((df["balls_before"] >= 3) & (df["strikes_before"] >= 2)).astype(int)
    df["count_diff"] = df["strikes_before"] - df["balls_before"]
    df["count_total"] = df["strikes_before"] + df["balls_before"]

    df["win_exp_diff"] = df["home_win_expectancy"] - df["away_win_expectancy"]
    df["abs_score_diff_pitcher"] = df["score_diff_pitcher_team"].abs()
    df["late_and_close"] = ((df["inning"] >= 8) & (df["abs_score_diff_pitcher"] <= 1)).astype(int)

    df["is_high_leverage"] = (df["li"] >= 1.5).astype(int)
    df["li_count_diff"] = df["li"] * df["count_diff"]
    df["li_late_close"] = df["li"] * df["late_and_close"]

    df["same_hand"] = (df["pitcher_hand"] == df["batter_hand"]).astype(int)

    df["pitcher_cold_start"] = df["asof_pitcher_n"].fillna(0).eq(0).astype(int)
    df["batter_cold_start"] = df["asof_batter_n"].fillna(0).eq(0).astype(int)
    df["pitchmix_cold_start"] = df["asof_pitcher_pitchmix_n"].fillna(0).eq(0).astype(int)

    def shrink(rate_col, n_col, k=30):
        n = df[n_col].fillna(0)
        r = df[rate_col].fillna(global_mean)
        return (n * r + k * global_mean) / (n + k)

    df["pitcher_success_rate_smooth"] = shrink("asof_pitcher_success_rate", "asof_pitcher_n")
    df["batter_success_rate_smooth"] = shrink("asof_batter_success_rate", "asof_batter_n")
    df["matchup_success_diff"] = df["pitcher_success_rate_smooth"] - df["batter_success_rate_smooth"]
    df["matchup_middle_diff"] = (
        df["asof_pitcher_middle_rate"].fillna(global_mean)
        - df["asof_batter_middle_rate"].fillna(global_mean)
    )

    df["pitcher_recent_trend"] = df["asof_pitcher_prev1_game_success_rate"] - df["asof_pitcher_prev5_game_success_rate"]
    df["pitcher_recent_trend3"] = df["asof_pitcher_prev3_game_success_rate"] - df["asof_pitcher_prev5_game_success_rate"]

    fb = df["asof_pitcher_fastball_rate"].fillna(0)
    br = df["asof_pitcher_breaking_rate"].fillna(0)
    os_ = df["asof_pitcher_offspeed_rate"].fillna(0)
    df["pitchmix_max_share"] = np.maximum.reduce([fb, br, os_])

    df["month_sin"] = np.sin(2 * np.pi * df["game_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["game_month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["game_dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["game_dayofweek"] / 7)

    tk_keys = ["balls_before", "strikes_before", "outs_before"]
    tk_cols = [c for c in tk_lookup.columns if c.startswith("tk_")]
    tk_lookup = tk_lookup[tk_keys + tk_cols].copy()
    for k in tk_keys:
        tk_lookup[k] = tk_lookup[k].astype(df[k].dtype)
    orig_index = df.index
    df = df.merge(
        tk_lookup, on=tk_keys, how="left",
        validate="many_to_one", sort=False,
    )
    df.index = orig_index
    df["tk_fastball_dev"] = fb - df["tk_fastball_rate"]
    df["tk_breaking_dev"] = br - df["tk_breaking_rate"]
    df["tk_offspeed_dev"] = os_ - df["tk_offspeed_rate"]

    return df.drop(columns=[ID_COL], errors="ignore")

## 3. 제출 파일 생성 유틸

제출 파일은 `sample_submission.csv` 와 **같은 row_id 순서, 같은 컬럼**이어야 합니다.
모델 예측을 `row_id` 기준으로 `sample_submission` 에 채워 넣습니다. 예측에 없는
`row_id` 는 기존 placeholder 값을 유지합니다.

In [3]:
# =======================
# 제출 파일 생성 유틸
# =======================

def merge_predictions(sub, ids, preds):
    """sample_submission의 row_id 순서에 맞춰 예측 확률 병합.

    예측에 없는 row_id는 sample_submission의 기존 값(placeholder)을 유지.
    """
    pred_map = dict(zip(ids, preds))
    values, n_missing = [], 0
    for rid, cur in zip(sub[ID_COL], sub[TARGET_COL]):
        p = pred_map.get(rid)
        if p is None:
            n_missing += 1
            values.append(cur)
        else:
            values.append(p)
    if n_missing:
        print(f" 경고: 예측이 없어 placeholder를 유지한 row_id {n_missing}건")
    sub[TARGET_COL] = values
    return sub


def save_submission(path, sub):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    sub.to_csv(path, index=False, encoding="utf-8")

## 4. 추론 실행

학습 노트북이 저장한 LightGBM/CatBoost 앙상블과 로지스틱 회귀 메타러너(스태킹)를
불러와 평가 데이터를 예측하고 `./output/submission.csv` 를 생성합니다.

각 모델의 `predict_proba(X)[:, 1]` 을 2개 피처로 하는 `stack_model.predict_proba(...)`
로 결합합니다(v10 — 이전 버전의 가중 블렌드 + isotonic 보정을 대체). 제출값은 0 이상
1 이하의 실수여야 하므로 0/1 로 반올림하지 않습니다.

In [ ]:
# =======================
# main
# =======================
# 이 노트북은 notebooks/ 아래에 있으므로 경로가 "../data" 등이다. script.py(src/)로
# 코드를 옮길 때는 제출 zip 안에서 model/, script.py가 같은 위치이므로 "./data" 로
# 접두사를 되돌려야 한다 — 이미 src/script.py 에는 그렇게 반영되어 있다.

def main():
    # ---- 경로 변수 (필요에 따라 수정) ----
    TEST_DIR = "../data"           # test.csv, sample_submission.csv 위치
    MODEL_DIR = "../model"         # ensemble.pkl 위치
    OUT_DIR = "../output"
    TEST_PATH = os.path.join(TEST_DIR, "test.csv")
    SAMPLE_SUB_PATH = os.path.join(TEST_DIR, "sample_submission.csv")
    MODEL_PATH = os.path.join(MODEL_DIR, "ensemble.pkl")
    OUT_PATH = os.path.join(OUT_DIR, "submission.csv")

    # ---- 모델 아티팩트 로드 (LightGBM + CatBoost + 로지스틱 회귀 메타러너(스태킹) + 피처 목록 + global_mean + tk_lookup) ----
    print("Load model...")
    artifact = joblib.load(MODEL_PATH)
    lgbm_model = artifact["lgbm_model"]
    cb_model = artifact["catboost_model"]
    stack_model = artifact["stack_model"]
    cat_cols = artifact["cat_cols"]
    all_features = artifact["all_features"]
    global_mean = artifact["global_mean"]
    tk_lookup = artifact["tk_lookup"]
    print(f" OK. n_features={len(all_features)}")

    # ---- 테스트 데이터 로드 ----
    print("Load test data...")
    test = load_test(TEST_PATH)
    sub = load_sample_submission(SAMPLE_SUB_PATH)
    print(f" test={len(test)}  submission={len(sub)}")

    # ---- 전처리 & 피처 엔지니어링 (학습과 동일) ----
    print("Build features...")
    ids = test[ID_COL].tolist()
    X = build_features(test, global_mean, tk_lookup)
    X = X[all_features]
    for c in cat_cols:
        X[c] = X[c].astype("category")
    X_cb = X.copy()
    for c in cat_cols:
        X_cb[c] = X_cb[c].astype(str)
    print(f" features={X.shape[1]}")

    # ---- 예측 (LightGBM/CatBoost) + 로지스틱 회귀 메타러너 스태킹 ----
    # v10: 기존 "블렌드 가중치 격자탐색 + isotonic 보정"을 LightGBM/CatBoost 예측 2개를
    # 입력으로 하는 로지스틱 회귀 메타러너로 교체했다(학습 노트북 섹션 6/7 참고).
    # 2024 홀드아웃에서 메타러너의 raw predict_proba가 그 위에 isotonic을 한 번 더
    # 씌운 경우보다 더 좋게 나와(782.47 vs 770.21) 추가 보정 없이 그대로 쓴다.
    print("Inference model...")
    if len(X):
        p_lgb = lgbm_model.predict_proba(X)[:, 1]
        p_cb = cb_model.predict_proba(X_cb)[:, 1]
        X_meta = np.column_stack([p_lgb, p_cb])
        preds = stack_model.predict_proba(X_meta)[:, 1]
    else:
        preds = []
    print(f" preds={len(preds)}")

    # ---- sample_submission 기반 결과 생성 ----
    print("Build submission...")
    sub = merge_predictions(sub, ids, preds)
    save_submission(OUT_PATH, sub)
    print(f"Saved: {OUT_PATH} (rows={len(sub)})")


if __name__ == "__main__":
    main()